## cleaning bills data

In [1]:
# imports
import pandas as pd

In [ ]:
# data downloaded from online, 2500 bills in bills1.csv, bills2.csv, bills3.csv, ...
bills1_df = pd.read_csv(
    "../data/bills/bills1.csv",
    skiprows=2
)

# concatenating bills dataframes of 2500 bills each into bills_df
# ...

# edit this so that it reads in all of the bills123456...csvs and concatenates them together
# bills_df = ...

## investigations

In [ ]:
# convert to datetime format
date_cols = [
    "Date of Introduction",
    "Latest Action Date",
    "Date Offered",
    "Date Submitted",
    "Date Proposed"
]
# date range (work in progress, need to get more data)
for col in date_cols:
    bills_df[col] = pd.to_datetime(bills_df[col], errors="coerce")

print(f'min date: {bills_df["Date of Introduction"].min()}')
print(f'max date: {bills_df["Date of Introduction"].max()}')

In [ ]:
# missing vals
missing = bills_df.isnull().sum().sort_values(ascending=False)
print(missing)

In [ ]:
# List of columns to drop
drop_cols = [
    "Date Offered",
    "Date Submitted",
    "Date Proposed",
    "Amends Bill",
    "Amends Amendment"
]

bills_df = bills_df.drop(columns=drop_cols)
bills_df.columns

In [ ]:
# most frequent committees
print(bills_df["Committees"].value_counts().head())

# most frequent sponsors
print(bills_df["Sponsor"].value_counts().head())

# most frequent latest action
bills_df["Latest Action"].value_counts().head()

In [ ]:
# misc
print(f'df.shape: {bills_df.shape}')
print(f'df.columns: {bills_df.columns}')
print(f'df.info(): {bills_df.info()}')
print(f'df.describe(include="all"): {bills_df.describe(include="all")}')

## cleaning the sponsor name and their info

In [ ]:
# parse sponsor name, party, and state
def parse_sponsor(s):
    match = re.match(r"^(.*?)\s+\[(?:Rep\.|Sen\.|Del\.)-([A-Z])-([A-Z]{2})-", str(s))
    if match:
        name, party_code, state = match.groups()
        party = "Democrat" if party_code == "D" else "Republican" if party_code == "R" else "Independent"
        # split "Last, First" → handle multi-word last names like "Watson Coleman, Bonnie"
        if "," in name:
            last, first = name.split(",", 1)
            last = last.strip()
            first = first.strip()
        else:
            last, first = name.strip(), None
        return first, last, party, state
    return None, s, None, None

bills_df[["Sponsor First Name", "Sponsor Last Name", "Party", "State"]] = bills_df["Sponsor"].apply(
    lambda x: pd.Series(parse_sponsor(x))
)

# drop sponsor col
bills_df = bills_df.drop(columns=["Sponsor"])

# make date of introduction and latest action date into datetime
bills_df["Date of Introduction"] = pd.to_datetime(bills_df["Date of Introduction"])
bills_df["Latest Action Date"] = pd.to_datetime(bills_df["Latest Action Date"])

# strip "House - " or "Senate - " prefix from Committees
bills_df["Committees"] = bills_df["Committees"].str.replace(r"^(House|Senate)\s*-\s*", "", regex=True)

bills_df.head()

topics, clustering, sentence embeddings, etc. would come next

In [ ]:
# another way

In [3]:
import requests
import pandas as pd
import time

API_KEY  = 'OtWuMddeA7AEheFMyj7zPCqtN7YTJB69CjhGYjpy'
BASE_URL = 'https://api.congress.gov/v3'

def get_bills_page(offset, limit=250):
    url = (
        f'{BASE_URL}/bill/119'
        f'?api_key={API_KEY}&format=json'
        f'&limit={limit}&offset={offset}'
        f'&sort=updateDate+desc'
    )
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json().get('bills', [])


all_bills = []
offset    = 0
target    = 25000

while len(all_bills) < target:
    batch = get_bills_page(offset)
    if not batch:
        break
    all_bills.extend(batch)
    print(f'  fetched {len(all_bills)} bills...')
    offset += 250
    time.sleep(0.5)

print(f'Done. {len(all_bills)} bills total.')

  fetched 250 bills...
  fetched 500 bills...
  fetched 750 bills...
  fetched 1000 bills...
  fetched 1250 bills...
  fetched 1500 bills...
  fetched 1750 bills...
  fetched 2000 bills...
  fetched 2250 bills...
  fetched 2500 bills...
  fetched 2750 bills...
  fetched 3000 bills...
  fetched 3250 bills...
  fetched 3500 bills...
  fetched 3750 bills...
  fetched 4000 bills...
  fetched 4250 bills...
  fetched 4500 bills...
  fetched 4750 bills...
  fetched 5000 bills...
  fetched 5250 bills...
  fetched 5500 bills...
  fetched 5750 bills...
  fetched 6000 bills...
  fetched 6250 bills...
  fetched 6500 bills...
  fetched 6750 bills...
  fetched 7000 bills...
  fetched 7250 bills...
  fetched 7500 bills...
  fetched 7750 bills...
  fetched 8000 bills...
  fetched 8250 bills...
  fetched 8500 bills...
  fetched 8750 bills...
  fetched 9000 bills...
  fetched 9250 bills...
  fetched 9500 bills...
  fetched 9750 bills...
  fetched 10000 bills...
  fetched 10250 bills...
  fetched 10500 b

In [5]:
rows = []
for b in all_bills:
    sponsor     = b.get('sponsors', [{}])[0] if b.get('sponsors') else {}
    latest      = b.get('latestAction') or {}   # handles both missing key and None value
    rows.append({
        'Legislation Number':   f"{b.get('type', '')}. {b.get('number', '')}",
        'Title':                b.get('title'),
        'Date of Introduction': b.get('introducedDate'),
        'Latest Action':        latest.get('text'),
        'Latest Action Date':   latest.get('actionDate'),
        'Sponsor First Name':   sponsor.get('firstName'),
        'Sponsor Last Name':    sponsor.get('lastName'),
        'Party':                sponsor.get('party'),
        'State':                sponsor.get('state'),
        'Congress':             b.get('congress'),
        'URL':                  b.get('url', '').replace('api.congress.gov/v3', 'congress.gov')
    })

bills_df = pd.DataFrame(rows)
bills_df.head()

,Legislation Number,Title,Date of Introduction,Latest Action,Latest Action Date,Sponsor First Name,Sponsor Last Name,Party,State,Congress,URL
0,S. 2255,Trafficking Survivors Relief Act of 2025,None,Read twice and referred to the Committee on th...,2025-07-10,None,None,None,None,119,https://congress.gov/bill/119/s/2255?format=json
1,HR. 4323,Trafficking Survivors Relief Act,None,Became Public Law No: 119-73.,2026-01-23,None,None,None,None,119,https://congress.gov/bill/119/hr/4323?format=json
2,S. 2726,Beautifying Federal Civic Architecture Act of ...,None,Read twice and referred to the Committee on En...,2025-09-04,None,None,None,None,119,https://congress.gov/bill/119/s/2726?format=json
3,HR. 5194,Beautifying Federal Civic Architecture Act of ...,None,Referred to the Subcommittee on Economic Devel...,2025-09-09,None,None,None,None,119,https://congress.gov/bill/119/hr/5194?format=json
4,HRES. 702,Condemning in the strongest possible terms the...,None,Referred to the House Committee on Oversight a...,2025-09-11,None,None,None,None,119,https://congress.gov/bill/119/hres/702?format=...


got 14761 of the most recent bills, and took about 4 mins to scrape --> next step is ideally to get more than that, maybe 50k bills???